In [177]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt 
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.linear_model import LinearRegression
import os

In [178]:
# Cargamos los csv de los tifs
path = "saved_files/dataset"
dfs = {}
for archivo in os.listdir(path):
    if archivo.endswith("_features.csv"):
        nombre_sin_extension = os.path.splitext(archivo)[0]  # sin .csv
        ruta_completa = os.path.join(path, archivo)
        dfs[nombre_sin_extension] = pd.read_csv(ruta_completa)

Como hay algunos días y puntos concretos en los que las reflectancias son 0 y los índices por tanto se han calculado como NaN, limpiamos los datasets de esas filas:

In [179]:
for nombre_df, df in dfs.items():
    for band_set in ["rhow", "rhown"]:
        dfs[nombre_df] = df.dropna()

In [171]:
dfs["c2x-complex-nets_1x1_imida_depth_gt_1_features"].head(2)

,Date,Buoy,Latitude,Longitude,rhow_B1,rhow_B2,rhow_B3,rhow_B4,rhow_B5,rhow_B6,...,dif_rel_4bands_rhown_B3_B4_B5_B1,dif_rel_4bands_rhown_B3_B4_B5_B2,dif_rel_4bands_rhown_B3_B5_B4_B1,dif_rel_4bands_rhown_B3_B5_B4_B2,dif_rel_4bands_rhown_B4_B1_B5_B2,dif_rel_4bands_rhown_B4_B1_B5_B3,dif_rel_4bands_rhown_B4_B2_B5_B1,dif_rel_4bands_rhown_B4_B2_B5_B3,dif_rel_4bands_rhown_B4_B3_B5_B1,dif_rel_4bands_rhown_B4_B3_B5_B2
0,2017-06-30,CTD1,4187246,695025,0.023568,0.034184,0.043664,0.014988,0.009146,0.00232,...,2.434,2.563,3.829,4.033,0.367,0.428,0.033,0.224,-0.063,0.067
1,2017-06-30,CTD2,4181518,693105,0.013951,0.022364,0.026365,0.006800,0.003868,0.00100,...,3.431,3.546,5.623,5.812,0.311,0.339,0.006,0.150,-0.039,0.077


Las fórmulas que queremos comprobar son:

- $y_1 = 0.6386 e^{4.7513x}$, donde $y$ es la clorofila y $x$ es la relación de $(Green - Blue)/(Green + Blue)$

- $y_2 = 124.94x - 115.35$ donde $x = \frac{Green+NIR1}{Green + Red}$

- $y_3 = 22.835x - 12.974$ donde $x = \frac{NIR1 - NIR2}{Red - NIR2}$

- $y_4 = 32.448x - 21.408$ donde $x = \frac{NIR1}{Red}$


con las fechas


dates = [
    "28/10/2016", "20/06/2018", "07/11/2018", "14/08/2019",
    "30/06/2017", "10/07/2018", "12/03/2019", "18/09/2019",
    "20/02/2018", "29/08/2018", "25/06/2019", "03/10/2019",
    "07/03/2018", "03/10/2018"
]


Filtramos los dataframes para dejar solamente esas fechas:

In [180]:
dates = [
    "28/10/2016", "20/06/2018", "07/11/2018", "14/08/2019",
    "30/06/2017", "10/07/2018", "12/03/2019", "18/09/2019",
    "20/02/2018", "29/08/2018", "25/06/2019", "03/10/2019",
    "07/03/2018", "03/10/2018"
]
filter_dates = pd.to_datetime(dates, format="%d/%m/%Y")

for nombre_df, df in dfs.items():
    df["Date"] = pd.to_datetime(df["Date"], format="%Y-%m-%d", errors='coerce')  # asegurarse
    dfs[nombre_df] = df[df["Date"].isin(filter_dates)].copy()


Recordemos que en el paper quitan las observaciones cuya medida de Chl-a esté fuera de la mediana $\pm$ una desviación estándar. Para ello hacemos lo siguiente (no lo hacemos porque los resultados empeoran muchísimo.). De hacerlo, pasaríamos de tener 130 filas, a quedarnos solamente con 107.

In [69]:
# for nombre_df, df in dfs.items():
#     if "Chl" in df.columns:
#         mediana = df["Chl"].median()
#         desviacion = df["Chl"].std()
#         umbral_sup = mediana + desviacion
#         umbral_inf = mediana - desviacion

#         # Filtrar
#         df_filtrado = df[(df["Chl"] >= umbral_inf) & (df["Chl"] <= umbral_sup)].copy()
#         dfs[nombre_df] = df_filtrado

Definimos funciones para las fórmulas que queremos aplicar.
Recordar que:
- Blue = rhow_B2
- Green = rhow_B3
- Red = rhow_B4
- NIR1 = rhow_B5
- NIR2 = rhow_B6

In [182]:
def add_y(blue, green, red, nir1, nir2):
    x_1 = (green-blue)/(green+blue)
    y_1 = 0.6386*np.exp(4.7513*x_1)

    x_2 = (green+nir1)/(green+red)
    y_2 = 124.94*x_2 - 115.35

    x_3 = (nir1-nir2)/(red-nir2)
    y_3 = 22.835*x_3 - 12.974

    x_4 = nir1/red
    y_4 = 32.448*x_4 - 21.408

    return y_1, y_2, y_3, y_4

In [183]:
def add_y_columns_to_df(df, band_set="rhow"):
    # Mapas de bandas
    mapping = band_maps[band_set]

    # Extraer bandas desde el DataFrame
    blue = df[mapping["Blue"]]
    green = df[mapping["Green"]]
    red = df[mapping["Red"]]
    nir1 = df[mapping["NIR1"]]
    nir2 = df[mapping["NIR2"]]

    # Aplicar fórmula
    y_1, y_2, y_3, y_4 = add_y(blue, green, red, nir1, nir2)

    # Añadir columnas al DataFrame
    if band_set == "rhow":
        df["y_1_rhow"] = y_1
        df["y_2_rhow"] = y_2
        df["y_3_rhow"] = y_3
        df["y_4_rhow"] = y_4

    elif band_set == "rhown":
        df["y_1_rhown"] = y_1
        df["y_2_rhown"] = y_2
        df["y_3_rhown"] = y_3
        df["y_4_rhown"] = y_4

    return df

band_maps = {
    "rhow": {
        "Blue": "rhow_B2",
        "Green": "rhow_B3",
        "Red": "rhow_B4",
        "NIR1": "rhow_B5",
        "NIR2": "rhow_B6"
    },
    "rhown": {
        "Blue": "rhown_B2",
        "Green": "rhown_B3",
        "Red": "rhown_B4",
        "NIR1": "rhown_B5",
        "NIR2": "rhown_B6"
    }
}

In [184]:
for nombre_df, df in dfs.items():
    for band_set in ["rhow", "rhown"]:
        dfs[nombre_df] = add_y_columns_to_df(df, band_set)

In [202]:
metrics_df = pd.DataFrame()
for nombre_df, df in dfs.items():
    
    # R2 (coeficiente de determinación)
    try:
        metrics_df.loc[nombre_df[:-9], "r2_y_1_rhow"] = r2_score(df["Chl"], df["y_1_rhow"]).round(3)
        metrics_df.loc[nombre_df[:-9], "r2_y_2_rhow"] = r2_score(df["Chl"], df["y_2_rhow"]).round(3)
        metrics_df.loc[nombre_df[:-9], "r2_y_3_rhow"] = r2_score(df["Chl"], df["y_3_rhow"]).round(3)
        metrics_df.loc[nombre_df[:-9], "r2_y_4_rhow"] = r2_score(df["Chl"], df["y_4_rhow"]).round(3)
    except:
        continue

    try:
        metrics_df.loc[nombre_df[:-9], "r2_y_1_rhown"] = r2_score(df["Chl"], df["y_1_rhown"]).round(3)
        metrics_df.loc[nombre_df[:-9], "r2_y_2_rhown"] = r2_score(df["Chl"], df["y_2_rhown"]).round(3)
        metrics_df.loc[nombre_df[:-9], "r2_y_3_rhown"] = r2_score(df["Chl"], df["y_3_rhown"]).round(3)
        metrics_df.loc[nombre_df[:-9], "r2_y_4_rhown"] = r2_score(df["Chl"], df["y_4_rhown"]).round(3)
    except:
        continue

    # Coeficiente de correlación (correlación lineal)
    # try:
    #     metrics_df.loc[nombre_df[:-9], "corr_y_1_rhow"] = df["Chl"].corr(df["y_1_rhow"]).round(3)
    #     metrics_df.loc[nombre_df[:-9], "corr_y_2_rhow"] = df["Chl"].corr(df["y_2_rhow"]).round(3)
    #     metrics_df.loc[nombre_df[:-9], "corr_y_3_rhow"] = df["Chl"].corr(df["y_3_rhow"]).round(3)
    #     metrics_df.loc[nombre_df[:-9], "corr_y_4_rhow"] = df["Chl"].corr(df["y_4_rhow"]).round(3)
    # except:
    #     continue

    # try:
    #     metrics_df.loc[nombre_df[:-9], "corr_y_1_rhown"] = df["Chl"].corr(df["y_1_rhown"]).round(3)
    #     metrics_df.loc[nombre_df[:-9], "corr_y_2_rhown"] = df["Chl"].corr(df["y_2_rhown"]).round(3)
    #     metrics_df.loc[nombre_df[:-9], "corr_y_3_rhown"] = df["Chl"].corr(df["y_3_rhown"]).round(3)
    #     metrics_df.loc[nombre_df[:-9], "corr_y_4_rhown"] = df["Chl"].corr(df["y_4_rhown"]).round(3)
    # except:
    #     continue

    # RMSE
    # try:
    # metrics_df.loc[nombre_df[:-9], "rmse_y_1_rhow"] = np.sqrt(mean_squared_error(df["Chl"], df["y_1_rhow"])).round(3)
    # metrics_df.loc[nombre_df[:-9], "rmse_y_2_rhow"] = np.sqrt(mean_squared_error(df["Chl"], df["y_2_rhow"])).round(3)
    # metrics_df.loc[nombre_df[:-9], "rmse_y_3_rhow"] = np.sqrt(mean_squared_error(df["Chl"], df["y_3_rhow"])).round(3)
    # metrics_df.loc[nombre_df[:-9], "rmse_y_4_rhow"] = np.sqrt(mean_squared_error(df["Chl"], df["y_4_rhow"])).round(3)
    # except:
    # continue

    # try:
    # metrics_df.loc[nombre_df[:-9], "rmse_y_1_rhown"] = np.sqrt(mean_squared_error(df["Chl"], df["y_1_rhown"])).round(3)
    # metrics_df.loc[nombre_df[:-9], "rmse_y_2_rhown"] = np.sqrt(mean_squared_error(df["Chl"], df["y_2_rhown"])).round(3)
    # metrics_df.loc[nombre_df[:-9], "rmse_y_3_rhown"] = np.sqrt(mean_squared_error(df["Chl"], df["y_3_rhown"])).round(3)
    # metrics_df.loc[nombre_df[:-9], "rmse_y_4_rhown"] = np.sqrt(mean_squared_error(df["Chl"], df["y_4_rhown"])).round(3)
    # except:
    # continue

    # #NRMSE
    # try:
    # metrics_df.loc[nombre_df[:-9], "nrmse_y_1_rhow"] = (metrics_df.loc[nombre_df[:-9], "rmse_y_1_rhow"]/(df["Chl"].max()-df["Chl"].min())).round(3)*100
    # metrics_df.loc[nombre_df[:-9], "nrmse_y_2_rhow"] = (metrics_df.loc[nombre_df[:-9], "rmse_y_2_rhow"]/(df["Chl"].max()-df["Chl"].min())).round(3)*100
    # metrics_df.loc[nombre_df[:-9], "nrmse_y_3_rhow"] = (metrics_df.loc[nombre_df[:-9], "rmse_y_3_rhow"]/(df["Chl"].max()-df["Chl"].min())).round(3)*100
    # metrics_df.loc[nombre_df[:-9], "nrmse_y_4_rhow"] = (metrics_df.loc[nombre_df[:-9], "rmse_y_4_rhow"]/(df["Chl"].max()-df["Chl"].min())).round(3)*100
    # except:
    # continue

In [205]:
filtered_df = metrics_df.loc[metrics_df.index.str.contains("imida")]
filtered_df[filtered_df.gt(0.73).any(axis=1)]

,r2_y_1_rhow,r2_y_2_rhow,r2_y_3_rhow,r2_y_4_rhow,r2_y_1_rhown,r2_y_2_rhown,r2_y_3_rhown,r2_y_4_rhown
c2x-nets_3x3_imida_depth_eq_0,0.191,0.732,0.765,0.720,0.227,0.528,0.586,0.599
c2x-nets_5x5_imida_depth_lt_1,0.199,0.698,0.738,0.694,0.238,0.527,0.586,0.604
c2x-nets_1x1_imida_depth_lt_1,0.178,0.709,0.754,0.688,0.221,0.519,0.579,0.583
c2x-nets_1x1_imida_depth_eq_0,0.177,0.717,0.748,0.694,0.216,0.494,0.548,0.559
c2x-nets_5x5_imida_depth_eq_0,0.196,0.717,0.744,0.707,0.233,0.505,0.558,0.581
c2x-nets_3x3_imida_depth_lt_1,0.192,0.713,0.763,0.708,0.232,0.554,0.619,0.625


In [206]:
filtered_df = metrics_df.loc[metrics_df.index.str.contains("upct")]
filtered_df[filtered_df.gt(0.73).any(axis=1)]

,r2_y_1_rhow,r2_y_2_rhow,r2_y_3_rhow,r2_y_4_rhow,r2_y_1_rhown,r2_y_2_rhown,r2_y_3_rhown,r2_y_4_rhown
c2x-nets_9x9_upct_depth_lt_2,0.229,0.677,0.749,0.694,0.269,0.604,0.671,0.674
c2x-nets_3x3_upct_depth_eq_1,0.186,0.762,0.782,0.736,0.222,0.544,0.592,0.605
c2x-nets_9x9_upct_depth_eq_1,0.162,0.745,0.755,0.721,0.192,0.509,0.551,0.573
c2x-nets_5x5_upct_depth_lt_2,0.242,0.686,0.769,0.704,0.289,0.631,0.701,0.703
c2x-nets_5x5_upct_depth_eq_1,0.191,0.750,0.766,0.729,0.228,0.524,0.568,0.590
c2x-nets_3x3_upct_depth_lt_2,0.235,0.657,0.751,0.680,0.284,0.635,0.717,0.705
c2x-nets_3x3_upct_depth_lt_1,0.226,0.520,0.651,0.564,0.282,0.648,0.737,0.703
c2x-nets_1x1_upct_depth_eq_1,0.180,0.729,0.750,0.700,0.219,0.496,0.545,0.558


In [210]:
dfs["c2x-nets_3x3_upct_depth_lt_1_features"]["Chl"]

0      0.549
1      0.441
2      0.617
3      0.629
4      0.269
       ...  
195    3.417
196    2.323
197    2.830
198    2.527
199    2.106
Name: Chl, Length: 130, dtype: float64

In [215]:
from sklearn.linear_model import LinearRegression

df = dfs["c2x-nets_3x3_upct_depth_lt_1_features"][["y_1_rhow", "y_2_rhow","y_3_rhow","y_4_rhow","Chl"]]
X = df[["y_1_rhow", "y_2_rhow","y_3_rhow","y_4_rhow"]]
y = df["Chl"]

model = LinearRegression().fit(X, y)
pred = model.predict(X)

r2 = r2_score(y, pred)
rmse = np.sqrt(mean_squared_error(y, pred))
nrmse = rmse / (np.max(y)- np.min(y)) * 100
print('RMSE: %.3f' % rmse)
print('R squared: %.3f' % r2)
print(f"NRMSE: {nrmse:.1f}%")

RMSE: 1.405
R squared: 0.888
NRMSE: 7.8%
